# 01-2 · Restaurant and member taste vectors

| | |
|---|---|
| **What it does** | (a) averages dish vectors up to one taste vector per restaurant, (b) loads the simulated group members with their own taste vectors |
| **Proves** | F0 — every member has a real taste vector, not a missing field. This was the paper's headline blocker until it was fixed |
| **Ported from** | Vincent (`wangvincentx-cmd`), commit `3c32c76` "Add per-member F0 taste vectors and C1 fairness aggregation selectors" (2026-08-02) — the persona fixture, `resolve_taste_vector`, and the neutral fallback |
| **Ported from** | Vihaan, branch `research-tasks`, `usertasteembeddings.py` (commit `f045279`, 2026-07-17) — the 95 simulated taste vectors Vincent deduped into the fixture below. The original file also pushed those vectors to a Pinecone index using a hardcoded API key; **that key is not reproduced here** — see PRD-0, it must be rotated |
| **Eval surface** | `resolve_taste_vector` — flip a zero vector and watch it resolve to neutral instead of scoring 0 against everything |

No API key needed.


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)


In [2]:
import nbio, json
nbio.bootstrap()


repo     : /project/painting/local-projects-to-run/research/ai-in-food/fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/local-projects-to-run/research/ai-in-food/fair-mediator')

## Step 1 — restaurant taste vectors, from the dish vectors in 01-1

Re-derive the same dish taste vectors (deterministic, so this is not duplicated
logic — it is the same function applied again) and average up to one vector per
restaurant, using only dishes that scored non-zero.


In [3]:
# Step 1: rebuild dish taste vectors, then average up to restaurant level
import csv

flavor_map = {}
with open(nbio.DATA / "ingredient-flavor.csv") as f:
    for row in csv.DictReader(f):
        name = row.get("ingredient", "").strip().lower()
        if name:
            flavor_map[name] = {d: float(row.get(d, 0) or 0) for d in
                                 ["sweet", "salty", "sour", "bitter", "umami", "spicy"]}

def infer_taste(dish_name):
    matched = [flavor_map[ing] for ing in flavor_map if ing in dish_name.lower()]
    if not matched:
        return [0.0] * 6
    return [sum(m[d] for m in matched) / len(matched) for d in nbio.TASTE_DIMS]

def iter_dish_names(node):
    if isinstance(node, dict):
        if ("item" in node or "name" in node) and "price" in node:
            yield (node.get("item") or node.get("name") or "").strip()
            return
        for v in node.values():
            yield from iter_dish_names(v)
    elif isinstance(node, list):
        for v in node:
            yield from iter_dish_names(v)

restaurants = {}
for f in sorted((nbio.DATA / "Menu").glob("*.json")):
    payload = json.loads(f.read_text())
    dishes = [n for n in iter_dish_names(payload) if n]
    vectors = [infer_taste(d) for d in dishes]
    nonzero = [v for v in vectors if sum(abs(x) for x in v) > 0]
    avg = [sum(v[i] for v in nonzero) / len(nonzero) for i in range(6)] if nonzero else [0.0] * 6
    restaurants[f.stem] = {"n_dishes": len(dishes), "n_nonzero": len(nonzero), "taste": avg}

nbio.table([{"restaurant": name[:34], **{"n_dishes": r["n_dishes"], "n_tasted": r["n_nonzero"]}}
            for name, r in restaurants.items()])


restaurant              n_dishes  n_tasted
------------------------------------------
Arabian Corner Restaur  60        24      
Casa.cafe               129       62      
Delicia Multi Cuisine   155       77      
House of Chefs          198       93      
Ministry of South Bar   32        11      


## Step 2 — the simulated group members (Vihaan's vectors, Vincent's fixture)

95 simulated members, each with a diet, an allergy set, and a taste vector. This
is `_PERSONAS` from `group_session_service.py`, harvested with the source data
intact and the leaked key left behind.


In [4]:
# Step 2: load the harvested persona fixture
personas = json.load(open(nbio.DATA / "personas.json"))
print(f"personas: {len(personas)}")
for p in personas[:5]:
    print(f"   {p['display_name']:<10} diet={p['diet_type']:<8} allergies={p['allergies']}")
    print(nbio.taste(p["taste_vector"]))


personas: 95
   Alice      diet=non-veg  allergies=['peanuts']
    sweet #######..... 0.60
    salty #####....... 0.40
    sour  ##.......... 0.20
    bitte #........... 0.10
    umami ###########. 0.90
    spicy ##########.. 0.80
   Vihaan     diet=veg      allergies=[]
    sweet #######..... 0.60
    salty ######...... 0.50
    sour  ########.... 0.70
    bitte ####........ 0.30
    umami #####....... 0.40
    spicy ##########.. 0.80
   Kyle       diet=non-veg  allergies=['shellfish']
    sweet #........... 0.10
    salty ######...... 0.50
    sour  #........... 0.10
    bitte #####....... 0.40
    umami #........... 0.10
    spicy #........... 0.10
   Vincent    diet=vegan    allergies=['dairy']
    sweet ####........ 0.30
    salty ##.......... 0.20
    sour  #######..... 0.60
    bitte #........... 0.10
    umami ######...... 0.50
    spicy #####....... 0.40
   Manisha    diet=vegan    allergies=['dairy']
    sweet #####....... 0.40
    salty ##########.. 0.80
    sour  ##........

## Step 3 — `resolve_taste_vector`: the fix that unblocked F0

Before this fix, a member with an all-zero taste vector scored 0.0 against every
restaurant — not "no preference", but "worst possible preference", which
silently penalised new or anonymous members. Flip one to demonstrate.


In [5]:
# Step 3: the neutral-fallback demonstration
zero_member = [0.0] * 6
print("raw vector       :", zero_member)
print("resolved         :", nbio.resolve_taste_vector(zero_member))
print()
real_member = personas[0]["taste_vector"]
print("a real vector is returned unchanged:", nbio.resolve_taste_vector(real_member) == real_member)


raw vector       : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
resolved         : [0.5, 0.5, 0.5, 0.5, 0.5, 0.5]

a real vector is returned unchanged: True


### Reading it

Restaurants now have taste vectors (Step 1) and so do members (Step 2), and a
member with no signal resolves to neutral rather than to zero (Step 3) — this is
the fix that made scoring possible at all. Notice already, from the restaurant
table above, that all five menu-backed restaurants score dishes from the same
narrow set of cuisines: this is the seed of a finding that becomes concrete in
**03-aggregate/03** — the fairness dial has very little room to move on a catalog
this small and this similar.

Next: **02-safety** builds the feasible set F from these same members.
